## Exercícios

> Retirados de [learn-python: sqlalchemy_orm-questions](https://aviadr1.github.io/learn-advanced-python/11_db_access/exercise/sqlalchemy_orm-questions.html).

#### Q1.

Baixa e extraia o arquivo compactado com o banco de dados [Chinook database](https://www.sqlitetutorial.net/sqlite-sample-database/). Salve o arquivo `chinook.db` na mesma pasta deste script.
* Link para baixar: http://www.sqlitetutorial.net/wp-content/uploads/2018/03/chinook.zip

<img width=500 src=https://www.sqlitetutorial.net/wp-content/uploads/2015/11/sqlite-sample-database-color.jpg>


#### Q2.

Leia o código e os comentários das células a seguir para entender como acessamos os modelos ORM de um banco já existente.

In [ ]:
from sqlalchemy import create_engine, text, MetaData
from sqlalchemy.orm import Session

engine = create_engine("sqlite+pysqlite:///chinook.db", echo=False)

### extrai as classes da base de dados Chinook
metadata = MetaData()
metadata.reflect(engine)

# O metadata tem informações sobre as tabelas
# que serão usadas para criar os modelos ORM
for table_name, table in metadata.tables.items():
    print(table_name)
    print(table.columns.keys())
    print(table.columns.items())
    print('-'*25)

### configura o objeto Base mapeando os modelos ORM das tabelas
from sqlalchemy.ext.automap import automap_base
Base = automap_base(metadata=metadata)
Base.prepare()

# o objeto Base tem os modelos ORM que podemos usar
# para manipular o banco de dados
print(Base.classes.items())

In [ ]:
# A seguir um exemplo de query na tabela Albums
# usamos o objeto Base para acessar cada modelo ORM.

session = Session(engine)
res = session.scalars(select(Base.classes.albums))
first_album = res.first()
print(first_album.AlbumId, first_album.Title)

#### Q3. 
Com base nos códigos anteriores realize as operações solicitadas nas células a seguir:


In [ ]:
# Imprima os três primeiros registros da tabela tracks
stmt1 = select(Track).limit(3)
resultados1 = session.scalars(stmt1)

for track in resultados1:
    print(f"ID: {track.TrackId} | Nome: {track.Name} | Compositor: {track.Composer}")

In [ ]:
# Imprima o nome da faixa e o título do álbum das primeiras 20 faixas na tabela tracks.
stmt2 = select(Track.Name, Album.Title).join(Album).limit(20)
resultados2 = session.execute(stmt2)

for track_name, album_title in resultados2:
    print(f"Faixa: {track_name} | Álbum: {album_title}")

In [ ]:
# Imprima as 10 primeiras vendas de faixas da tabela invoice_items
# Para essas 10 primeiras vendas, imprima os nomes das faixas vendidas e a quantidade vendida.
stmt3 = select(Track.Name, InvoiceItem.Quantity).join(Track).limit(10)
resultados3 = session.execute(stmt3)

for track_name, qty in resultados3:
    print(f"Faixa: {track_name} | Quantidade vendida: {qty}")

In [ ]:
# Imprima os nomes das 10 faixas mais vendidas e quantas vezes foram vendidas.
stmt4 = (
    select(Track.Name, func.sum(InvoiceItem.Quantity).label('total_vendido'))
    .join(InvoiceItem)
    .group_by(Track.TrackId)
    .order_by(desc('total_vendido'))
    .limit(10)
)
resultados4 = session.execute(stmt4)

print("--- TOP 10 FAIXAS ---")
for track_name, total in resultados4:
    print(f"Faixa: {track_name} | Total Vendido: {total}")

In [ ]:
# Quem são os 10 artistas que mais venderam?
# dica: você precisa juntar as tabelas invoice_items, tracks, albums e artists
stmt5 = (
    select(Artist.Name, func.sum(InvoiceItem.Quantity).label('total_vendido'))
    .join(Album, Artist.ArtistId == Album.ArtistId) # Join Artista -> Álbum
    .join(Track, Album.AlbumId == Track.AlbumId)    # Join Álbum -> Faixa
    .join(InvoiceItem, Track.TrackId == InvoiceItem.TrackId) # Join Faixa -> Venda
    .group_by(Artist.ArtistId)
    .order_by(desc('total_vendido'))
    .limit(10)
)
resultados5 = session.execute(stmt5)

print("--- TOP 10 ARTISTAS ---")
for artist_name, total in resultados5:
    print(f"Artista: {artist_name} | Total Vendido: {total}")